# Tiny Miles: RL Training with SGLang

In this lab, you will build a **minimal Reinforcement Learning (RL) training loop** using SGLang as the inference engine. We extract essential components from [Miles](https://github.com/your-org/miles), the native RL framework for SGLang, to create "Tiny Miles" — a simplified but functional GRPO training pipeline that improves a language model's math reasoning on GSM8K.

You will learn how to use SGLang for RL rollouts with logprob returns, train with PyTorch FSDP and GRPO, synchronize weights live between training and inference engines, and manage GPU memory for colocated training via `torch_memory_saver` and CPU offloading — all orchestrated in an end-to-end training loop from this notebook.

## Architecture

```
┌──────────────────────────────────────────────────────────────┐
│                    Notebook (Scheduler)                       │
│                                                              │
│  ┌─ Rollout Phase ──────────────────────────────────┐        │
│  │  SGLang on GPU  (KV cache + weights active)      │        │
│  │  FSDP on CPU    (model + optimizer offloaded)     │        │
│  └───────────────────────────────────────────────────┘        │
│                          ↓                                    │
│            release_memory_occupation()                        │
│            wake_up() — FSDP model → GPU                      │
│                          ↓                                    │
│  ┌─ Train Phase ────────────────────────────────────┐        │
│  │  FSDP on GPU    (forward/backward/optimizer)      │        │
│  │  SGLang paused  (KV cache + weights released)     │        │
│  └───────────────────────────────────────────────────┘        │
│                          ↓                                    │
│            update_weights() — FSDP → SGLang                  │
│            sleep() — FSDP model → CPU                        │
│            resume_memory_occupation()                         │
│                          ↓                                    │
│                    next Rollout                               │
└──────────────────────────────────────────────────────────────┘
```

- **Inference Engine**: SGLang server (TP=2 across both GPUs, with `torch_memory_saver`)
- **Training Engine**: FSDP server via `torchrun` (both GPUs, with CPU offload between steps)
- **Scheduler**: This notebook — orchestrates the training loop and memory transitions

Both engines are **colocated** on the same 2× H100 GPUs and **time-share GPU memory**: during rollout, SGLang owns the GPU; during training, FSDP owns the GPU. The `torch_memory_saver` library enables this by dynamically releasing and re-allocating SGLang's GPU memory (`cuMemRelease`/`cuMemCreate`), while the training server offloads its model and optimizer state to CPU RAM between training steps.

---
## 0. Prerequisites

Ensure `sglang`, `torch`, `transformers`, `flask`, and `datasets` are installed.


In [1]:
!pip install --ignore-installed blinker flask
!pip install pylatexenc accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.5/226.5 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=9a3641da5e7575056e57f8e82751a99800908a1ccff804e0de1d29a65dd6be94
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [3]:
import os
import sys
import json
import time
import asyncio
import subprocess

import requests
import httpx
import numpy as np
from transformers import AutoTokenizer

---
## 1. Configuration

In [4]:
# Download Qwen3-0.6B if not already present
from huggingface_hub import snapshot_download

MODEL_PATH = os.environ.get("MODEL_PATH", "/root/models/Qwen3-0.6B")

if not os.path.exists(MODEL_PATH):
    print(f"Downloading Qwen3-0.6B to {MODEL_PATH}...")
    snapshot_download(repo_id="Qwen/Qwen3-0.6B", local_dir=MODEL_PATH)
    print("Download complete!")
else:
    print(f"Model already exists at {MODEL_PATH}")

# SGLang Inference Engine
SGLANG_HOST = "127.0.0.1"
SGLANG_PORT = 30000
SGLANG_URL = f"http://{SGLANG_HOST}:{SGLANG_PORT}"

# FSDP Training Engine
TRAIN_HOST = "127.0.0.1"
TRAIN_PORT = 5000
TRAIN_URL = f"http://{TRAIN_HOST}:{TRAIN_PORT}"

# RL Hyperparameters
BATCH_SIZE = 16               # Unique prompts per rollout iteration
N_SAMPLES_PER_PROMPT = 8      # Responses per prompt (for GRPO group normalization)
ROLLOUT_TEMPERATURE = 1.0     # Sampling temperature for diversity
MAX_NEW_TOKENS = 1024         # Max response length
LR = 5e-6                     # Learning rate
EPS_CLIP = 0.2                # PPO clipping epsilon
NUM_ITERATIONS = 40           # Number of training iterations

# Micro-batching: split the rollout batch into smaller chunks for the forward/backward pass.
# This trades iteration time for peak GPU memory. The training result is identical to
# processing the full batch at once (gradients are accumulated and scaled correctly).
# Rule of thumb: MICRO_BATCH_SIZE × MAX_NEW_TOKENS ≲ 16k tokens per micro-batch.
MICRO_BATCH_SIZE = 4          # Samples per micro-batch (BATCH_SIZE*N_SAMPLES / MICRO_BATCH_SIZE micro-batches)

# SGLang TP size (for weight sync)
TP_SIZE = 2

print(f"Model: {MODEL_PATH}")
print(f"SGLang URL: {SGLANG_URL}")
print(f"Training URL: {TRAIN_URL}")
print(f"Batch size: {BATCH_SIZE} prompts × {N_SAMPLES_PER_PROMPT} samples = {BATCH_SIZE * N_SAMPLES_PER_PROMPT} total")
print(f"Micro-batch size: {MICRO_BATCH_SIZE} ({BATCH_SIZE * N_SAMPLES_PER_PROMPT // MICRO_BATCH_SIZE} micro-batches per step)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Download complete!
Model: /root/models/Qwen3-0.6B
SGLang URL: http://127.0.0.1:30000
Training URL: http://127.0.0.1:5000
Batch size: 16 prompts × 8 samples = 128 total
Micro-batch size: 4 (32 micro-batches per step)


---
## 2. Launch SGLang Server (Inference Engine)

We launch SGLang with **TP=2** across both H100 GPUs. Key flags:
- `--mem-fraction-static 0.3` — limits KV cache allocation to 30% of GPU memory
- `--enable-memory-saver` — activates `torch_memory_saver`, which uses CUDA's virtual memory API (`cuMemCreate`/`cuMemRelease`) to dynamically release and re-allocate SGLang's GPU memory. This allows the training engine to reclaim SGLang's memory during training steps via the `/release_memory_occupation` and `/resume_memory_occupation` HTTP endpoints.

This mirrors `miles/backends/sglang_utils/sglang_engine.py`, which sets `enable_memory_saver=True` when colocated training is enabled (`--offload-rollout`).

In [5]:
SGLANG_CMD = (
    f"python3 -m sglang.launch_server "
    f"--model-path {MODEL_PATH} "
    f"--tp {TP_SIZE} "
    f"--host {SGLANG_HOST} --port {SGLANG_PORT} "
    f"--mem-fraction-static 0.3 "
    f"--enable-memory-saver "
    f"--trust-remote-code"
)
print(f"Launch command:\n{SGLANG_CMD}")

Launch command:
python3 -m sglang.launch_server --model-path /root/models/Qwen3-0.6B --tp 2 --host 127.0.0.1 --port 30000 --mem-fraction-static 0.3 --enable-memory-saver --trust-remote-code


In [6]:
# Clean up any leftover servers from previous runs
import subprocess, time, socket

def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

def _kill_servers():
    for pattern in ["sglang", "fsdp_training_server"]:
        subprocess.run(f"pkill -9 -f {pattern} || true", shell=True, stderr=subprocess.DEVNULL)

    for port in [SGLANG_PORT, TRAIN_PORT]:
        subprocess.run(
            f"lsof -ti:{port} | xargs -r kill -9 || true",
            shell=True, stderr=subprocess.DEVNULL
        )

    time.sleep(5)

_kill_servers()

In [11]:
import pathlib

_log_dir = pathlib.Path("logs")
_log_dir.mkdir(exist_ok=True)
_sglang_log_path = _log_dir / "sglang_server.log"
_sglang_log_file = open(_sglang_log_path, "w")

sglang_proc = subprocess.Popen(
    SGLANG_CMD,
    shell=True,
    stdout=_sglang_log_file,
    stderr=_sglang_log_file,
    text=True,
)
print(f"SGLang server log: {_sglang_log_path.resolve()}")

SGLang server log: /content/logs/sglang_server.log


In [12]:
from sglang.utils import wait_for_server

wait_for_server(SGLANG_URL, timeout=300)
print("SGLang server is ready!")

ModuleNotFoundError: No module named 'sglang'

In [ ]:
# Sanity check: generate a simple response
import openai

client = openai.Client(base_url=f"{SGLANG_URL}/v1", api_key="EMPTY")
response = client.chat.completions.create(
    model=MODEL_PATH,
    messages=[{"role": "user", "content": "What is 2+2?"}],
    temperature=0,
    max_tokens=64,
)
print(f"SGLang response: {response.choices[0].message.content}")

---
## 3. Launch FSDP Training Server (Training Engine)

The training server runs as a separate process via `torchrun`. It initializes the **same model** with FSDP (Fully Sharded Data Parallel) and exposes HTTP endpoints for training, weight updates, and GPU memory management.

Since FSDP requires `dist.init_process_group`, it cannot run inside a Jupyter notebook — hence the external server pattern. This mirrors the Miles architecture where training and inference are separate engines coordinated by a scheduler.

### HTTP Endpoints

| Endpoint | Purpose |
|---|---|
| `GET /health` | Health check |
| `POST /train_step` | Receive rollout data, perform one GRPO training step |
| `POST /update_weights` | Sync weights to SGLang inference engine |
| `POST /wake_up` | Load model + optimizer from CPU → GPU before training |
| `POST /sleep` | Offload model + optimizer from GPU → CPU after training |

The `sleep`/`wake_up` cycle mirrors `miles/backends/fsdp_utils/actor.py` (`--offload-train` mode), freeing GPU memory for SGLang during rollout.

In [ ]:
TRAIN_CMD = (
    f"torchrun --nproc-per-node={TP_SIZE} "
    f"fsdp_training_server.py "
    f"--model-path {MODEL_PATH} "
    f"--sglang-url {SGLANG_URL} "
    f"--port {TRAIN_PORT} "
    f"--lr {LR} "
    f"--eps-clip {EPS_CLIP} "
    f"--tp-size {TP_SIZE} "
    f"--micro-batch-size {MICRO_BATCH_SIZE} "
    f"--gradient-checkpointing"
)
print(f"Launch command:\n{TRAIN_CMD}")

In [ ]:
import pathlib

_log_dir = pathlib.Path("logs")
_log_dir.mkdir(exist_ok=True)
_train_log_path = _log_dir / "train_server.log"
_train_log_file = open(_train_log_path, "w")

train_proc = subprocess.Popen(
    TRAIN_CMD,
    shell=True,
    stdout=_train_log_file,
    stderr=_train_log_file,
    text=True,
    cwd=os.path.dirname(os.path.abspath("fsdp_training_server.py")),
)
print(f"Training server log: {_train_log_path.resolve()}")

In [ ]:
# Wait for training server to be ready
for i in range(120):
    try:
        resp = requests.get(f"{TRAIN_URL}/health", timeout=5)
        if resp.status_code == 200:
            print(f"Training server is ready! {resp.json()}")
            break
    except Exception:
        pass
    time.sleep(2)
    if i % 10 == 0:
        print(f"Waiting for training server... ({i*2}s)")
else:
    print("ERROR: Training server failed to start. Check logs:")
    print(train_proc.stdout.read())

---
## 4. Data Preparation (GSM8K)

We use **GSM8K** (Grade School Math 8K), a dataset of math word problems. Each problem has a question and a numeric answer.

The data loader mirrors `miles/rollout/data_source.py: RolloutDataSource`, which:
1. Loads the dataset
2. Formats prompts using the model's chat template
3. Creates groups of `n_samples_per_prompt` copies for GRPO

In [ ]:
from gsm8k_utils import load_gsm8k, GSM8KDataLoader

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)

train_data = load_gsm8k(split="train")
test_data = load_gsm8k(split="test")

print(f"Train: {len(train_data)} problems")
print(f"Test:  {len(test_data)} problems")
print(f"\nExample:")
print(f"  Question: {train_data[0]['question'][:200]}...")
print(f"  Answer:   {train_data[0]['label']}")

In [ ]:
data_loader = GSM8KDataLoader(
    train_data,
    tokenizer,
    batch_size=BATCH_SIZE,
    n_samples_per_prompt=N_SAMPLES_PER_PROMPT,
)
print(f"DataLoader: {len(data_loader)} prompts, batch_size={BATCH_SIZE}, n_samples={N_SAMPLES_PER_PROMPT}")

---
## 5. Rollout Phase

The rollout phase generates responses from the current policy (SGLang inference engine). For each prompt, we generate `N_SAMPLES_PER_PROMPT` responses with `return_logprob=True` to get per-token log probabilities.

This mirrors `miles/rollout/sglang_rollout.py: generate()`, which:
1. Sends tokenized prompts to SGLang's `/generate` endpoint
2. Collects response text, token IDs, and log probabilities
3. Packages the data for reward computation and training

The log probabilities from the inference engine serve as the **old policy** $\pi_{\text{old}}$ in the PPO importance ratio.

In [ ]:
from tqdm.asyncio import tqdm as atqdm

async def rollout_batch(sglang_url, batch, temperature, max_new_tokens):
    """Generate responses for a batch of prompt groups.

    Mirrors miles/rollout/sglang_rollout.py: generate().

    Args:
        sglang_url: SGLang server URL
        batch: list of groups, each group is a list of sample dicts
        temperature: sampling temperature
        max_new_tokens: max response length

    Returns:
        list of response dicts with keys:
            full_token_ids, rollout_log_probs, response_text,
            prompt_length, response_length, label
    """
    results = []

    async with httpx.AsyncClient(timeout=httpx.Timeout(None)) as client:
        tasks = []
        sample_meta = []  # Track metadata for each request

        for group in batch:
            for sample in group:
                payload = {
                    "input_ids": sample["token_ids"],
                    "sampling_params": {
                        "temperature": temperature,
                        "max_new_tokens": max_new_tokens,
                        "skip_special_tokens": True,
                    },
                    "return_logprob": True,
                }
                tasks.append(client.post(f"{sglang_url}/generate", json=payload))
                sample_meta.append(sample)

        # Execute all requests concurrently with progress bar
        responses = await atqdm.gather(*tasks, desc="Rollout", total=len(tasks))

    for resp, sample in zip(responses, sample_meta):
        output = resp.json()
        meta = output.get("meta_info", {})

        # Extract log probs for response tokens
        # SGLang returns: output_token_logprobs = [(logprob, token_id, token_text), ...]
        output_logprobs = meta.get("output_token_logprobs", [])
        rollout_log_probs = [item[0] for item in output_logprobs]

        # Extract response token IDs
        response_token_ids = [item[1] for item in output_logprobs]

        results.append({
            "full_token_ids": sample["token_ids"] + response_token_ids,
            "rollout_log_probs": rollout_log_probs,
            "response_text": output.get("text", ""),
            "prompt_length": len(sample["token_ids"]),
            "response_length": len(response_token_ids),
            "label": sample["label"],
        })

    return results

In [ ]:
# Test rollout with a single batch
batch = data_loader.get_batch()
responses = await rollout_batch(SGLANG_URL, batch, ROLLOUT_TEMPERATURE, MAX_NEW_TOKENS)

print(f"Generated {len(responses)} responses")
print(f"\nExample response:")
print(f"  Prompt length:   {responses[0]['prompt_length']} tokens")
print(f"  Response length: {responses[0]['response_length']} tokens")
print(f"  Text: {responses[0]['response_text'][:300]}...")
print(f"  Log probs (first 5): {responses[0]['rollout_log_probs'][:5]}")

---
## 6. Reward Computation

For GSM8K, we use a **rule-based reward** that checks if the model's answer matches the ground truth. The reward function is extracted from `miles/rollout/rm_hub/math_utils.py: grade_answer_verl()`.

It extracts the answer from `\boxed{}` notation and uses both strict string matching and SymPy algebraic equivalence for comparison.

- **Reward = 1.0** if the extracted answer matches the ground truth
- **Reward = 0.0** otherwise

In [ ]:
from reward_utils import grade_answer_verl

# Compute rewards for all responses
rewards = []
for r in responses:
    is_correct = grade_answer_verl(r["response_text"], r["label"])
    rewards.append(1.0 if is_correct else 0.0)

mean_reward = sum(rewards) / len(rewards)
print(f"Mean reward: {mean_reward:.3f} ({sum(r == 1.0 for r in rewards)}/{len(rewards)} correct)")

In [ ]:
def filter_zero_std_groups(responses, rewards, n_samples_per_prompt):
    """Filter out prompt groups where all rewards are identical (std = 0).

    Mirrors miles/rollout/filter_hub/dynamic_sampling_filters.py: check_reward_nonzero_std().

    When every response for a prompt has the same reward, GRPO group-normalization
    produces zero advantage for every token in that group → zero gradient signal.
    Removing these groups before the training step avoids wasting forward/backward
    passes on examples that contribute nothing to learning.

    Common zero-std cases:
        • All N_SAMPLES responses correct  (easy problem) → std = 0, advantage = 0
        • All N_SAMPLES responses wrong    (hard problem) → std = 0, advantage = 0

    Args:
        responses: list of response dicts (length = n_groups * n_samples_per_prompt)
        rewards:   list of float, same length as responses
        n_samples_per_prompt: int

    Returns:
        (filtered_responses, filtered_rewards, n_filtered)
            filtered_* have all zero-std groups removed (still grouped in multiples of
            n_samples_per_prompt, so GRPO reshaping in the training server stays valid)
            n_filtered is the number of groups that were dropped
    """
    assert len(responses) % n_samples_per_prompt == 0, (
        f"len(responses)={len(responses)} not divisible by "
        f"n_samples_per_prompt={n_samples_per_prompt}"
    )
    n_groups = len(responses) // n_samples_per_prompt

    kept_responses = []
    kept_rewards = []
    n_filtered = 0

    for g in range(n_groups):
        start = g * n_samples_per_prompt
        end = start + n_samples_per_prompt
        group_rewards = rewards[start:end]

        # std > 0 iff not all rewards in the group are the same value
        if len(set(group_rewards)) > 1:
            kept_responses.extend(responses[start:end])
            kept_rewards.extend(group_rewards)
        else:
            n_filtered += 1

    return kept_responses, kept_rewards, n_filtered


# --- Test the filter on the batch we already have ---
filtered_responses, filtered_rewards, n_filtered = filter_zero_std_groups(
    responses, rewards, N_SAMPLES_PER_PROMPT
)
n_groups = len(responses) // N_SAMPLES_PER_PROMPT
n_kept = len(filtered_responses) // N_SAMPLES_PER_PROMPT
print(f"Groups: {n_groups} total → {n_kept} kept, {n_filtered} filtered (zero reward std)")
if filtered_rewards:
    print(f"Mean reward (kept groups): {sum(filtered_rewards) / len(filtered_rewards):.3f}")

---
## 7. Training Phase (GRPO)

Now we send the rollout data to the FSDP training server. The training step implements **GRPO (Group Relative Policy Optimization)**.

### How GRPO Works

For each prompt, we generate `n` responses and observe their rewards. GRPO normalizes rewards **within each prompt group**:

$$\text{advantage}_i = \frac{r_i - \mu_{\text{group}}}{\sigma_{\text{group}} + \epsilon}$$

This means:
- Responses with **above-average** reward get **positive** advantages → model increases their probability
- Responses with **below-average** reward get **negative** advantages → model decreases their probability

> **Note:** Advantages must be computed over the **full batch** before any splitting — the group normalization requires seeing all `N_SAMPLES_PER_PROMPT` rewards together.

### Gradient Accumulation over Micro-batches

The `BATCH_SIZE × N_SAMPLES_PER_PROMPT` sequences are too large to forward-pass all at once. Instead, the full batch is split into chunks of `MICRO_BATCH_SIZE` samples. For each chunk:
1. Forward pass → compute new log probs
2. Compute policy loss, scaled by `tokens_in_chunk / total_tokens`
3. `loss.backward()` — gradients **accumulate** without being zeroed

After all chunks, a single `optimizer.step()` is taken. The accumulated gradients are mathematically identical to what a full-batch pass would produce. This mirrors `miles/backends/fsdp_utils/actor.py: _train_core()` and `miles/backends/training_utils/data.py: DataIterator`.

### The Clipped Policy Loss

From `miles/utils/ppo_utils.py: compute_policy_loss()`:

$$\text{ratio} = \frac{\pi_\theta(a|s)}{\pi_{\text{old}}(a|s)} = \exp(\log\pi_\theta - \log\pi_{\text{old}})$$

$$L = \max\!\bigl(-\text{ratio} \cdot A,\ -\text{clip}(\text{ratio},\, 1-\epsilon,\, 1+\epsilon) \cdot A\bigr)$$

The clipping prevents the policy from changing too drastically in a single update (trust region).


In [ ]:
def package_rollout_data(responses, rewards, n_samples_per_prompt):
    """Package rollout results for the training server.

    Creates the data format expected by fsdp_training_server.py /train_step.
    """
    return {
        "tokens": [r["full_token_ids"] for r in responses],
        "rollout_log_probs": [r["rollout_log_probs"] for r in responses],
        "rewards": rewards,
        "prompt_lengths": [r["prompt_length"] for r in responses],
        "response_lengths": [r["response_length"] for r in responses],
        "n_samples_per_prompt": n_samples_per_prompt,
    }

In [ ]:
# Memory transition: Rollout → Train (same as training loop step 5)
requests.post(f"{SGLANG_URL}/release_memory_occupation", json={}, timeout=60)
requests.post(f"{TRAIN_URL}/wake_up", json={}, timeout=120)

# Send to training server
rollout_data = package_rollout_data(responses, rewards, N_SAMPLES_PER_PROMPT)

train_result = requests.post(f"{TRAIN_URL}/train_step", json=rollout_data, timeout=300)
train_metrics = train_result.json()

print("Training step result:")
for k, v in train_metrics.items():
    print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")

---
## 8. Weight Update & Memory Management

After training, we synchronize the updated model weights from the FSDP training engine to the SGLang inference engine, then transition GPU ownership back to SGLang.

### Weight Update

This uses SGLang's `/update_weights_from_tensor` HTTP endpoint. The training server:
1. Gathers the full state dict from FSDP (from `miles/backends/fsdp_utils/update_weight_utils.py`)
2. Groups parameters into size-bounded buckets
3. Serializes using `FlattenedTensorBucket` + `MultiprocessingSerializer` (IPC handles, not full data)
4. Sends to SGLang via HTTP — transfers are instant since both engines share the same GPUs

### GPU Memory Transitions

Since both engines are colocated on the same GPUs, they cannot both hold their full state in GPU memory simultaneously. The scheduler orchestrates memory transitions at each phase boundary:

| Transition | What Happens | API Calls |
|---|---|---|
| Rollout → Train | SGLang releases GPU memory (`cuMemRelease`), FSDP loads from CPU | `release_memory_occupation` → `wake_up` |
| Train → Rollout | FSDP offloads to CPU, SGLang re-allocates GPU memory (`cuMemCreate`) | `sleep` → `resume_memory_occupation` |

This mirrors the colocated approach in Miles:
- **SGLang side**: `miles/ray/rollout.py` calls `engine.release_memory_occupation()` / `engine.resume_memory_occupation()` (via `torch_memory_saver`)
- **FSDP side**: `miles/backends/fsdp_utils/actor.py` calls `sleep()` (`model.cpu()`) / `wake_up()` (`model.cuda()`) when `--offload-train` is enabled

In [ ]:
model_info = requests.get(f"{SGLANG_URL}/model_info").json()
print(f"Before update: SGLang weight version: {model_info.get('weight_version', 'N/A')}")

# Memory transition: Train → Weight sync (same as training loop step 7)
# Sleep FSDP first, then resume SGLang weights for the copy target
requests.post(f"{TRAIN_URL}/sleep", json={}, timeout=120)
requests.post(f"{SGLANG_URL}/resume_memory_occupation", json={"tags": ["weights"]}, timeout=60)

# Sync weights from training engine to inference engine
update_result = requests.post(
    f"{TRAIN_URL}/update_weights",
    json={"sglang_url": SGLANG_URL, "tp_size": TP_SIZE},
    timeout=300,
)
print(f"Weight update result: {update_result.json()}")

# Memory transition: Weight sync → Rollout (same as training loop step 8)
requests.post(f"{SGLANG_URL}/resume_memory_occupation", json={"tags": ["cuda_graph", "kv_cache"]}, timeout=60)

# Verify weight version on SGLang
model_info = requests.get(f"{SGLANG_URL}/model_info").json()
print(f"After update: SGLang weight version: {model_info.get('weight_version', 'N/A')}")

---
## 9. Full Training Loop

Before training, we establish a **baseline accuracy** using greedy evaluation on the test set. After `NUM_ITERATIONS` iterations, we evaluate again and compare the two numbers to see measurable improvement.

Each iteration:
```
1. Sample    : draw a batch of prompts from GSM8K
2. Rollout   : generate N_SAMPLES_PER_PROMPT responses per prompt via SGLang (with log probs)
3. Reward    : grade each response with grade_answer_verl() → 1.0 (correct) or 0.0 (wrong)
4. Filter    : drop groups where all rewards are identical — std=0 → zero GRPO advantage
5. Mem swap  : release SGLang GPU memory, wake up FSDP (CPU → GPU)
6. Train     : GRPO loss → gradient accumulation over micro-batches → optimizer step
7. Sync      : push updated weights from FSDP to SGLang
8. Mem swap  : sleep FSDP (GPU → CPU), resume SGLang GPU memory
9. Log       : record reward, loss, grad_norm, n_filtered, timing
```

Steps 5 and 8 are the **memory transitions** that enable colocated training:
- **Step 5** (Rollout → Train): SGLang's KV cache and weights are freed from GPU via `torch_memory_saver` (`cuMemRelease`), then FSDP's model and optimizer are loaded from CPU to GPU.
- **Step 8** (Train → Rollout): FSDP offloads to CPU, then SGLang re-allocates its memory on GPU (`cuMemCreate`).

This mirrors the outer loop in `miles/train.py`, with memory management from `miles/ray/rollout.py` (SGLang-side `release/resume_memory_occupation`) and `miles/backends/fsdp_utils/actor.py` (FSDP-side `sleep/wake_up`).

In [ ]:
from gsm8k_utils import format_prompt

EVAL_SIZE = 1319  # Number of test problems to evaluate on

async def evaluate(sglang_url, test_data, tokenizer, num_samples=EVAL_SIZE):
    """Evaluate model accuracy on the GSM8K test set.

    Uses greedy decoding (temperature=0) for deterministic results.
    Mirrors the evaluation loop from miles/rollout/sglang_rollout.py.

    Returns:
        (accuracy, n_correct, n_total)
    """
    eval_subset = test_data[:num_samples]

    async with httpx.AsyncClient(timeout=httpx.Timeout(None)) as client:
        tasks = []
        labels = []

        for item in eval_subset:
            formatted = format_prompt(item["question"], tokenizer)
            payload = {
                "input_ids": formatted["token_ids"],
                "sampling_params": {
                    "temperature": 0,          # greedy — deterministic
                    "max_new_tokens": MAX_NEW_TOKENS,
                    "skip_special_tokens": True,
                },
            }
            tasks.append(client.post(f"{sglang_url}/generate", json=payload))
            labels.append(item["label"])

        responses = await asyncio.gather(*tasks)

    correct = sum(
        grade_answer_verl(resp.json().get("text", ""), label)
        for resp, label in zip(responses, labels)
    )
    accuracy = correct / len(labels)
    return accuracy, correct, len(labels)


# ── Baseline evaluation ────────────────────────────────────────────────────
# Run BEFORE any training so we have a reference point to compare against.
# We reuse the same evaluate() function for the post-training measurement in
# Section 10, ensuring the comparison is apples-to-apples.
print(f"Baseline evaluation on {EVAL_SIZE} GSM8K test problems (greedy decoding)...")
initial_accuracy, initial_correct, initial_total = await evaluate(SGLANG_URL, test_data, tokenizer)
print(f"Baseline accuracy: {initial_correct}/{initial_total} = {initial_accuracy*100:.1f}%")

In [ ]:
def print_gpu_memory():
    """Print GPU memory usage for all GPUs via nvidia-smi."""
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,memory.used,memory.total,memory.free",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True
    )
    for line in result.stdout.strip().split("\n"):
        idx, used, total, free = [x.strip() for x in line.split(",")]
        print(f"  GPU {idx}: {used}/{total} MiB used ({free} MiB free)")

# Reset data loader for training
data_loader = GSM8KDataLoader(
    train_data,
    tokenizer,
    batch_size=BATCH_SIZE,
    n_samples_per_prompt=N_SAMPLES_PER_PROMPT,
)

metrics_history = []

for iteration in range(NUM_ITERATIONS):
    iter_start = time.time()

    # 1. Sample batch
    batch = data_loader.get_batch()

    # 2. Rollout (SGLang active on GPU, FSDP sleeping on CPU)
    responses = await rollout_batch(SGLANG_URL, batch, ROLLOUT_TEMPERATURE, MAX_NEW_TOKENS)
    rollout_time = time.time() - iter_start

    # 3. Compute rewards
    rewards = [
        1.0 if grade_answer_verl(r["response_text"], r["label"]) else 0.0
        for r in responses
    ]
    mean_reward = sum(rewards) / len(rewards)

    # 4. Filter groups with zero reward std (no GRPO signal)
    responses, rewards, n_filtered = filter_zero_std_groups(responses, rewards, N_SAMPLES_PER_PROMPT)
    if not responses:
        print(f"[Iter {iteration:3d}] All {BATCH_SIZE} groups filtered (zero reward std) — skipping")
        continue

    # 5. Memory transition: Rollout → Train
    #    Mirrors train.py:73-79: offload ALL SGLang memory, then wake up FSDP.
    requests.post(f"{SGLANG_URL}/release_memory_occupation", json={}, timeout=60)
    requests.post(f"{TRAIN_URL}/wake_up", json={}, timeout=120)

    # 6. Train
    rollout_data = package_rollout_data(responses, rewards, N_SAMPLES_PER_PROMPT)
    train_result = requests.post(
        f"{TRAIN_URL}/train_step", json=rollout_data, timeout=300
    ).json()
    train_time = time.time() - iter_start - rollout_time

    # 7. Memory transition: Train → Weight sync
    #    Mirrors train.py:92-95:
    #      offload_train()                          → sleep FSDP to CPU
    #      rollout_manager.onload_weights()          → resume SGLang weights only
    #      actor_model.update_weights()              → push weights (per-bucket .cuda())
    requests.post(f"{TRAIN_URL}/sleep", json={}, timeout=120)
    requests.post(f"{SGLANG_URL}/resume_memory_occupation", json={"tags": ["weights"]}, timeout=60)
    requests.post(
        f"{TRAIN_URL}/update_weights",
        json={"sglang_url": SGLANG_URL, "tp_size": TP_SIZE},
        timeout=300,
    )

    # 8. Memory transition: Weight sync → Rollout
    #    Mirrors train.py:96-97: rollout_manager.onload_kv()
    requests.post(f"{SGLANG_URL}/resume_memory_occupation", json={"tags": ["cuda_graph", "kv_cache"]}, timeout=60)
    total_time = time.time() - iter_start

    # 9. Log metrics
    metrics = {
        "iteration": iteration,
        "mean_reward": mean_reward,
        "n_filtered": n_filtered,
        "loss": train_result["loss"],
        "grad_norm": train_result["grad_norm"],
        "clipfrac": train_result["clipfrac"],
        "rollout_time": rollout_time,
        "train_time": train_time,
        "total_time": total_time,
    }
    metrics_history.append(metrics)

    print(
        f"[Iter {iteration:3d}] "
        f"reward={mean_reward:.3f}  "
        f"filtered={n_filtered}/{BATCH_SIZE}  "
        f"loss={train_result['loss']:.4f}  "
        f"grad_norm={train_result['grad_norm']:.4f}  "
        f"clip={train_result['clipfrac']:.3f}  "
        f"time={total_time:.1f}s (rollout={rollout_time:.1f}s train={train_time:.1f}s)"
    )

    # Print GPU memory every 10 iterations
    if (iteration + 1) % 10 == 0:
        print(f"--- GPU Memory at iteration {iteration} (rollout phase) ---")
        print_gpu_memory()

print("\nTraining complete!")
print("--- Final GPU Memory ---")
print_gpu_memory()

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

iters = [m["iteration"] for m in metrics_history]

axes[0].plot(iters, [m["mean_reward"] for m in metrics_history], "b-o")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("Mean Reward")
axes[0].set_title("Training Reward")
axes[0].grid(True)

axes[1].plot(iters, [m["loss"] for m in metrics_history], "r-o")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Loss")
axes[1].set_title("Policy Loss")
axes[1].grid(True)

axes[2].plot(iters, [m["grad_norm"] for m in metrics_history], "g-o")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Gradient Norm")
axes[2].set_title("Gradient Norm")
axes[2].grid(True)

plt.tight_layout()
plt.show()

---
## 10. Evaluation

We evaluate the trained model on the first `EVAL_SIZE` problems of the GSM8K test set using **greedy decoding** (`temperature=0`) for deterministic, reproducible results.

The baseline accuracy captured before training is compared against the post-training accuracy to measure how much GRPO improved the model on held-out problems.

In [ ]:
# --- Final evaluation (after training) ---
print(f"Evaluating trained model on {EVAL_SIZE} GSM8K test problems...")
final_accuracy, final_correct, final_total = await evaluate(SGLANG_URL, test_data, tokenizer)

delta_pp = (final_accuracy - initial_accuracy) * 100
print(f"\n{'='*45}")
print(f"  GSM8K Accuracy Results ({final_total} problems)")
print(f"{'='*45}")
print(f"  Before training : {initial_correct:3d}/{initial_total} = {initial_accuracy*100:5.1f}%")
print(f"  After  training : {final_correct:3d}/{final_total} = {final_accuracy*100:5.1f}%")
print(f"  Improvement     : {delta_pp:+.1f} pp")
print(f"{'='*45}")

---
## 11. Cleanup

In [ ]:
sglang_proc.terminate()
train_proc.terminate()
print("Servers terminated.")

---

### Going Further with Full Miles

**Enterprise-Grade Reinforcement Learning for Large-Scale Model Training** \
High-Performance Rollout • Low Precision Training • Production Stability

https://github.com/radixark/miles
